# Programme 7 — Normes matricielles et conditionnement

**Référence :** Kröger, *Numerische Mathematik 1 für AMP*, sections 2.1 et 2.2.

1. **Normes induites** : Zeilensummen- (Satz 2.5), Spalten- (Satz 2.7), Spektral- (Satz 2.9)
2. **Frobenius** et **Gesamtnorm** (Satz 2.10)
3. **Rayon spectral** $\rho(A) \leq \|A\|$ (Satz 2.14)
4. **Conditionnement** $\kappa(A) = \|A\| \cdot \|A^{-1}\|$ (Satz 2.17)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matrix_norms import (
    norm_inf_mat, norm_1_mat, norm_2_mat, norm_frobenius, norm_gesamt,
    rayon_spectral, condition,
    comparer_avec_numpy, verifier_submultiplicativite, verifier_borne_spectrale,
    tracer_condition_hilbert, tracer_borne_spectrale_aleatoire,
)

## 1. Les trois normes induites

| Norme vectorielle | Norme matricielle induite | Formule | Nom |
|---|---|---|---|
| $\|\cdot\|_\infty$ | $\|A\|_\infty = \max_i \sum_j \|a_{ij}\|$ | Satz 2.5 | Zeilensummennorm |
| $\|\cdot\|_1$ | $\|A\|_1 = \max_j \sum_i \|a_{ij}\|$ | Satz 2.7 | Spaltensummennorm |
| $\|\cdot\|_2$ | $\|A\|_2 = \sqrt{\lambda_{max}(A^* A)} = \sigma_{max}$ | Satz 2.9 | Spektralnorm |

**Astuce mnémotechnique du script :** appliquée à un vecteur colonne, la Spaltensummennorm donne la somme des composantes = norme 1. La Zeilensummennorm donne le max = norme ∞.

In [ ]:
# Übung 2.6
A = np.array([[3, 0, 2], [-4, 1+1j*np.sqrt(3), 2], [2j, -1, 1-1j]])
comparer_avec_numpy(A)

## 2. Submultiplikativité (Satz 2.2)

Toute norme matricielle satisfait $\|AB\| \leq \|A\| \cdot \|B\|$. C'est la propriété qui manque aux normes vectorielles et qui fait fonctionner l'analyse d'erreur.

In [ ]:
rng = np.random.default_rng(0)
B, C = rng.standard_normal((5, 5)), rng.standard_normal((5, 5))
for name, (lhs, rhs, ok) in verifier_submultiplicativite(B, C).items():
    print(f'{name}: ||BC|| = {lhs:.4f} ≤ ||B||·||C|| = {rhs:.4f}  {"✓" if ok else "✗"}')

## 3. Rayon spectral (Satz 2.14)

$\rho(A) \leq \|A\|$ pour **toute** norme matricielle. Et par Satz 2.15, la borne est serrée : pour tout $\varepsilon > 0$, il existe une norme telle que $\|A\| \leq \rho(A) + \varepsilon$.

In [ ]:
bornes = verifier_borne_spectrale(A)
rho = rayon_spectral(A)
print(f'ρ(A) = {rho:.4f}\n')
for name, (rho_val, norm_val) in bornes.items():
    print(f'  {rho_val:.4f} ≤ {name} = {norm_val:.4f}  ✓')

In [ ]:
tracer_borne_spectrale_aleatoire(n=5, n_matrices=300)
plt.show()

Tous les points sont sous la diagonale rouge — Satz 2.14 vérifié visuellement sur 300 matrices aléatoires.

## 4. Conditionnement (Satz 2.17)

$$\kappa(A) = \|A\| \cdot \|A^{-1}\|$$

C'est **le** nombre qui contrôle la propagation des erreurs dans $Ax = b$ :
$$\frac{\|\Delta x\|}{\|x\|} \leq \kappa(A) \cdot \frac{\|\Delta b\|}{\|b\|}$$

Si $\kappa(A) \sim 10^k$, on perd environ $k$ chiffres de précision.

In [ ]:
print(f'{"n":>3} | {"κ_∞(H_n) mine":>18} | {"κ_∞(H_n) numpy":>18} | {"chiffres perdus":>16}')
print('-' * 65)
for n in [3, 5, 8, 10, 12, 14]:
    i = np.arange(1, n+1)
    H = 1.0 / (i[:, None] + i[None, :] - 1)
    k = condition(H, np.inf)
    print(f'{n:>3} | {k:>18.2e} | {np.linalg.cond(H, np.inf):>18.2e} | {np.log10(k):>16.1f}')

In [ ]:
tracer_condition_hilbert()
plt.show()

Le conditionnement de Hilbert croît **exponentiellement** : chaque taille ajoutée multiplie $\kappa$ par environ $10^{2.5}$. À $n = 14$, $\kappa \sim 10^{18}$ : on a dépassé $1/\varepsilon_{mach}$, donc **toute** solution numérique est inutilisable.

## Récapitulatif

| Concept | Formule | Référence |
|---|---|---|
| Norme induite | $\|A\| = \sup \|Ah\|/\|h\|$ | Def. 2.1 |
| Submultiplikativité | $\|AB\| \leq \|A\|\|B\|$ | Satz 2.2 |
| $\rho(A) \leq \|A\|$ | pour toute norme | Satz 2.14 |
| $\kappa(A) = \|A\|\|A^{-1}\|$ | conditionnement | Satz 2.17 |

**Prochain programme :** `lu_decomposition.py` — la décomposition LR vue comme objet réutilisable pour résoudre $Ax = b$ avec multiples seconds membres.